# WM Prediction 2026 – Version 3

Erweiterungen gegenüber Version 2:

| Feature | Beschreibung |
|---|---|
| **Elo + Torabstand-Bonus** | Margin-of-Victory-Multiplikator: höhere K-Faktoren bei klaren Siegen |
| **Form der letzten 5 Spiele** | Gewichtete Punktzahl je Team als zusätzliches Modell-Feature |
| **Dixon-Coles-Korrektur** | Korrigiert die Poisson-Unabhängigkeit bei Niedrigtoren (0:0, 1:0, 0:1, 1:1) |

**Architektur (unverandert):**
- `all_matches` – alle Spiele seit 1872 → Elo-Berechnung
- `model_matches` – letzte 20 Jahre → Poisson-Training + Form-Feature

> Da bei einer WM alle Spiele auf neutralem Boden stattfinden, wird kein Heimvorteil modelliert.

## 1 · Setup

In [48]:
%pip install statsmodels scipy

import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy.stats import poisson
from scipy.optimize import minimize_scalar
import warnings
warnings.filterwarnings('ignore')

Defaulting to user installation because normal site-packages is not writeableNote: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2 · Daten laden

- `all_matches` – gesamter Datensatz (1872–heute), **nur für Elo**
- `model_matches` – letzte 20 Jahre, **für Poisson-Modell und Form-Feature**

In [49]:
RESULTS_FILE = "results.csv"

all_matches = pd.read_csv(RESULTS_FILE)
all_matches['date'] = pd.to_datetime(all_matches['date'])
all_matches = all_matches.sort_values('date').reset_index(drop=True)

cutoff = pd.Timestamp.now() - pd.DateOffset(years=20)
model_matches = all_matches[all_matches['date'] >= cutoff].copy()

print(f"Gesamt-Datensatz  (Elo):     {len(all_matches):>6} Spiele  "
      f"({all_matches['date'].dt.year.min()}–{all_matches['date'].dt.year.max()})")
print(f"Modell-Datensatz  (Poisson): {len(model_matches):>6} Spiele  "
      f"({model_matches['date'].dt.year.min()}–{model_matches['date'].dt.year.max()})")

Gesamt-Datensatz  (Elo):      49378 Spiele  (1872–2026)
Modell-Datensatz  (Poisson):  19345 Spiele  (2006–2026)


## 3 · Elo-Berechnung mit Torabstand-Bonus (gesamter Datensatz)

### Torabstand-Multiplikator
Klare Siege werden stärker gewichtet als knappe – analog zu Club-Elo und FIFA-Ranking:

$$m = \ln(\Delta g + 1) \cdot \frac{2.2}{\Delta E \cdot 0.001 + 2.2}$$

- $\Delta g$ = Tordifferenz (gecapped bei 1 für Unentschieden)
- $\Delta E$ = Elo-Differenz des Siegers vor dem Spiel (verhindert Inflation bei Favoritensiegen)
- Der zweite Term dämpft den Bonus wenn der Stärkere gewinnt

In [50]:
def get_k_factor(tournament):
    t = str(tournament)
    if t == "FIFA World Cup":            return 60
    if "UEFA Euro" in t:                 return 50
    if "Copa América" in t:              return 50
    if "African Cup of Nations" in t:    return 50
    if "AFC Asian Cup" in t:             return 50
    if "OFC Nations Cup" in t:           return 50
    if "Nations League" in t:            return 40
    if "qualification" in t.lower():     return 30
    if "Friendly" in t:                  return 15
    return 25


def mov_multiplier(goal_diff, elo_diff_winner):
    """Margin-of-Victory Multiplikator (goal_diff >= 0, elo_diff_winner >= 0)."""
    if goal_diff == 0:
        return 1.0
    return np.log(goal_diff + 1) * (2.2 / (elo_diff_winner * 0.001 + 2.2))


In [51]:
INITIAL_ELO = 1500
elo = {}

def get_elo(team):
    return elo.get(team, INITIAL_ELO)


elo_home_before = []
elo_away_before = []
current_year = None

for _, row in all_matches.iterrows():

    year = row['date'].year

    # Elo-Decay: 2 % Rückkehr Richtung Mittelwert pro Jahr
    if current_year is None:
        current_year = year
    if year > current_year:
        for team in elo:
            elo[team] = INITIAL_ELO + (elo[team] - INITIAL_ELO) * 0.98
        current_year = year

    home = row['home_team']
    away = row['away_team']
    eh   = get_elo(home)
    ea   = get_elo(away)

    elo_home_before.append(eh)
    elo_away_before.append(ea)

    expected_home = 1 / (1 + 10 ** ((ea - eh) / 400))

    hs, as_ = row['home_score'], row['away_score']
    goal_diff = abs(hs - as_)

    if hs > as_:
        actual_home = 1.0
        mov = mov_multiplier(goal_diff, max(eh - ea, 0))
    elif hs < as_:
        actual_home = 0.0
        mov = mov_multiplier(goal_diff, max(ea - eh, 0))
    else:
        actual_home = 0.5
        mov = 1.0

    k = get_k_factor(row['tournament'])
    elo[home] = eh + k * mov * (actual_home         - expected_home)
    elo[away] = ea + k * mov * ((1.0 - actual_home) - (1.0 - expected_home))


all_matches['elo_home'] = elo_home_before
all_matches['elo_away'] = elo_away_before
all_matches['elo_diff'] = all_matches['elo_home'] - all_matches['elo_away']

print(f"Elo-Berechnung abgeschlossen – {len(elo)} Teams im Verzeichnis.")

Elo-Berechnung abgeschlossen – 336 Teams im Verzeichnis.


## 4 · Aktuelle Elo-Tabelle

In [52]:
elo_table = (
    pd.DataFrame(list(elo.items()), columns=['team', 'elo'])
    .sort_values('elo', ascending=False)
    .reset_index(drop=True)
)
elo_table.index += 1
elo_table.head(25)

,team,elo
1,Spain,1942.993906
2,Argentina,1928.613089
3,France,1897.598626
4,England,1868.489810
5,Morocco,1864.639855
6,Portugal,1859.751425
7,Brazil,1854.294796
8,Colombia,1851.763156
9,Japan,1838.744895
10,Germany,1832.715057


## 5 · Form der letzten 5 Spiele

Für jedes Spiel berechnen wir den **gewichteten Formwert** jedes Teams *vor* diesem Spiel.

**Skala:** Sieg = 1.0 · Remis = 0.5 · Niederlage = 0.0 → `form_diff` ∈ [−1, 1]

| Gewicht | Spiel |
|---|---|
| 0.40 | letztes Spiel |
| 0.25 | vorletztes Spiel |
| 0.17 | 3. letztes |
| 0.11 | 4. letztes |
| 0.07 | 5. letztes |

> **Warum nicht 3/1/0-Punkte?** Mit Punkten läge `form_diff` im Bereich [−3, 3].
> Der GLM-Koeffizient würde dann dreimal so groß geschätzt und die Tordifferenz
> zwischen zwei Teams massiv übertreiben → Unentschieden würden kollabieren.
> Mit [0, 1]-Skala bleibt der Effekt proportional zu `elo_diff`.

In [53]:
FORM_WEIGHTS = np.array([0.40, 0.25, 0.17, 0.11, 0.07])  # jüngstes Spiel zuerst
FORM_N = len(FORM_WEIGHTS)


def match_points(score_for, score_against):
    """Ergebnis-Anteil [0, 1] aus Sicht des jeweiligen Teams (1=Sieg, 0.5=Remis, 0=Niederlage).
    Bewusst KEIN 3/1/0-Schema: damit bleibt form_diff im Bereich [-1, 1]
    und der GLM-Koeffizient wird nicht künstlich aufgeblasen."""
    if score_for > score_against:  return 1.0
    if score_for == score_against: return 0.5
    return 0.0


def weighted_form(dq):
    """Gewichteter Formwert in [0, 1] (jüngstes Spiel = höchstes Gewicht).
    Neutral-Startwert 0.5 = kein Vorteil für keine Seite."""
    pts = list(dq)
    if not pts:
        return 0.5          # Neutral: weder gute noch schlechte Form
    w = FORM_WEIGHTS[:len(pts)].copy()
    w /= w.sum()
    return float(np.dot(pts, w))


# Form über gesamten Datensatz chronologisch berechnen
from collections import defaultdict, deque

recent: dict = defaultdict(lambda: deque(maxlen=FORM_N))

form_home_list = []
form_away_list = []

for _, row in all_matches.iterrows():
    home = row['home_team']
    away = row['away_team']
    hs, as_ = row['home_score'], row['away_score']

    # Formwert VOR dem Spiel festhalten
    form_home_list.append(weighted_form(recent[home]))
    form_away_list.append(weighted_form(recent[away]))

    # Danach aktualisieren
    recent[home].appendleft(match_points(hs, as_))
    recent[away].appendleft(match_points(as_, hs))


all_matches['form_home'] = form_home_list
all_matches['form_away'] = form_away_list
all_matches['form_diff'] = all_matches['form_home'] - all_matches['form_away']

# Wertebereich prüfen
print("Form-Feature berechnet.")
print(f"form_diff Bereich: {all_matches['form_diff'].min():.2f} bis {all_matches['form_diff'].max():.2f}")
print(f"form_diff Mittel:  {all_matches['form_diff'].mean():.3f}  (sollte ~0 sein)")
all_matches[['home_team', 'away_team', 'date', 'form_home', 'form_away', 'form_diff']].tail(10)

Form-Feature berechnet.
form_diff Bereich: -1.00 bis 1.00
form_diff Mittel:  0.011  (sollte ~0 sein)


,home_team,away_team,date,form_home,form_away,form_diff
49368,Norway,France,2026-06-26,0.225,0.350,-0.125
49369,Egypt,Iran,2026-06-26,0.295,0.280,0.015
49370,New Zealand,Belgium,2026-06-26,0.110,0.295,-0.185
49371,Senegal,Iraq,2026-06-26,0.180,0.280,-0.100
49372,DR Congo,Uzbekistan,2026-06-27,0.265,0.125,0.140
49373,Colombia,Portugal,2026-06-27,0.170,0.295,-0.125
49374,Panama,England,2026-06-27,0.240,0.125,0.115
49375,Algeria,Austria,2026-06-27,0.295,0.350,-0.055
49376,Jordan,Argentina,2026-06-27,0.090,0.350,-0.260
49377,Croatia,Ghana,2026-06-27,0.070,0.085,-0.015


## 6 · Poisson-Modell (letzte 20 Jahre)

Features:
- `team` / `opponent` – Attack- und Defense-Koeffizienten
- `elo_diff` – Elo-Differenz aus Schritt 3 (historisch gewachsen)
- `form_diff` – Formunterschied aus Schritt 5 (letzte 5 Spiele)

In [54]:
# Alle Features aus all_matches in model_matches übernehmen
model_matches = model_matches.join(
    all_matches[['elo_home', 'elo_away', 'elo_diff', 'form_home', 'form_away', 'form_diff']],
    how='left'
)

### 6.1 · Turnier- und Zeitgewichtung

In [55]:
def tournament_weight(tournament):
    t = str(tournament)
    if 'FIFA World Cup' in t:           return 20
    if 'UEFA Euro' in t:                return 15
    if 'Copa América' in t:             return 15
    if 'African Cup of Nations' in t:   return 15
    if 'AFC Asian Cup' in t:            return 15
    if 'OFC Nations Cup' in t:          return 15
    if 'Nations League' in t:           return 8
    if 'qualification' in t.lower():    return 8
    if 'Friendly' in t:                 return 1
    return 2

model_matches['tournament_weight'] = model_matches['tournament'].apply(tournament_weight)

days_old = (pd.Timestamp.now() - model_matches['date']).dt.days
model_matches['time_weight'] = np.exp(-days_old / 1460)
model_matches['weight'] = model_matches['tournament_weight'] * model_matches['time_weight']

### 6.2 · Modell-Datensatz aufbauen

In [56]:
home_df = model_matches[['home_team', 'away_team', 'home_score',
                          'weight', 'elo_diff', 'form_diff']].copy()
home_df.columns = ['team', 'opponent', 'goals', 'weight', 'elo_diff', 'form_diff']

away_df = model_matches[['away_team', 'home_team', 'away_score',
                          'weight', 'elo_diff', 'form_diff']].copy()
away_df.columns = ['team', 'opponent', 'goals', 'weight', 'elo_diff', 'form_diff']
away_df['elo_diff']  = -away_df['elo_diff']   # Perspektive umkehren
away_df['form_diff'] = -away_df['form_diff']

goal_model_data = pd.concat([home_df, away_df], ignore_index=True)
print(f"Trainingszeilen: {len(goal_model_data):,}  "
      f"(= {len(goal_model_data) // 2:,} Spiele x 2 Perspektiven)")

Trainingszeilen: 38,690  (= 19,345 Spiele x 2 Perspektiven)


### 6.3 · Modell trainieren

In [57]:
model = smf.glm(
    formula='goals ~ team + opponent + elo_diff + form_diff',
    data=goal_model_data,
    family=sm.families.Poisson(),
    freq_weights=goal_model_data['weight']
).fit()

print(model.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:                  goals   No. Observations:                38546
Model:                            GLM   Df Residuals:                 76016.37
Model Family:                 Poisson   Df Model:                          633
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:            -1.0370e+05
Date:                Fri, 05 Jun 2026   Deviance:                       80539.
Time:                        14:48:43   Pearson chi2:                 7.26e+04
No. Iterations:                    20   Pseudo R-squ. (CS):             0.6965
Covariance Type:            nonrobust                                         
                                                   coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------------

## 7 · Dixon-Coles-Korrektur

Das Poisson-Modell nimmt an, dass Heim- und Auswaertstorre **unabhaengig** sind.
Empirisch sind Niedrigtor-Ergebnisse jedoch korreliert:

| Ergebnis | Poisson-Erwartung | Realitaet |
|---|---|---|
| 0:0 | zu selten | haeufiger |
| 1:1 | zu selten | haeufiger |
| 1:0 / 0:1 | zu haeufig | seltener |

Dixon & Coles (1997) korrigieren dies ueber einen **fixen** $\rho$-Parameter:

$$\tau(x, y, \mu_1, \mu_2, \rho) = \begin{cases}
1 - \mu_1 \mu_2 \rho & x=0, y=0 \\
1 + \mu_2 \rho & x=1, y=0 \\
1 + \mu_1 \rho & x=0, y=1 \\
1 - \rho & x=1, y=1 \\
1 & \text{sonst}
\end{cases}$$

> **Warum nicht optimiert?**  
> Die numerische MLE-Schaetzung von $\rho$ laeuft bei WM-Daten (neutraler Boden,
> kein Heimvorteil) an die Grenzwerte des Optimierers – das Ergebnis ist instabil.  
> Stattdessen verwenden wir $\rho = 0.07$, den in der Literatur am haeufigsten
> berichteten Wert fuer internationale Laenderspiele (Dixon & Coles 1997, Hvattum 2010).

In [58]:
# Korrektur-Faktor gemaess Dixon & Coles (1997)
# Vorzeichen gemaess Originalarbeit:
#   tau(0,0): 1 - mu1*mu2*rho  (0:0 wahrscheinlicher wenn rho > 0)
#   tau(1,0): 1 + mu2*rho      (Heimsieg 1:0, mu2 = Auswaerts-lambda)
#   tau(0,1): 1 + mu1*rho      (Auswaertssieg 0:1, mu1 = Heim-lambda)
#   tau(1,1): 1 - rho          (1:1 wahrscheinlicher wenn rho > 0)
def dixon_coles_tau(x, y, mu1, mu2, rho):
    if   x == 0 and y == 0: return 1.0 - mu1 * mu2 * rho
    elif x == 1 and y == 0: return 1.0 + mu2 * rho
    elif x == 0 and y == 1: return 1.0 + mu1 * rho
    elif x == 1 and y == 1: return 1.0 - rho
    return 1.0


# Fixer rho-Wert aus der Literatur fuer internationale Laenderspiele
# (Dixon & Coles 1997: ~0.06-0.09, hier konservativ 0.07)
# Positives rho bedeutet: 0:0 und 1:1 haeufiger, 1:0 und 0:1 seltener
# als vom unabhaengigen Poisson-Modell vorhergesagt
RHO = 0.07

# Sanity-Check: tau muss fuer alle relevanten Felder positiv sein
print("Dixon-Coles Korrektur-Faktoren bei mu=1.2, rho=0.07:")
for x, y in [(0,0),(1,0),(0,1),(1,1)]:
    t = dixon_coles_tau(x, y, 1.2, 1.2, RHO)
    print(f"  tau({x},{y}) = {t:.4f}  ('OK' if {t:.4f} > 0 else 'FEHLER')")

Dixon-Coles Korrektur-Faktoren bei mu=1.2, rho=0.07:
  tau(0,0) = 0.8992  ('OK' if 0.8992 > 0 else 'FEHLER')
  tau(1,0) = 1.0840  ('OK' if 1.0840 > 0 else 'FEHLER')
  tau(0,1) = 1.0840  ('OK' if 1.0840 > 0 else 'FEHLER')
  tau(1,1) = 0.9300  ('OK' if 0.9300 > 0 else 'FEHLER')


## 8 · Vorhersagefunktionen

In [59]:
def get_current_form(team):
    """Aktueller Formwert eines Teams in [0, 1]."""
    return weighted_form(recent[team])


def predict_goals(team, opponent):
    """Erwartete Tore von `team` gegen `opponent`.
    elo_diff  in ca. [-800, +800]
    form_diff in [-1, +1]"""
    elo_diff  = get_elo(team) - get_elo(opponent)
    form_diff = get_current_form(team) - get_current_form(opponent)
    df = pd.DataFrame({
        'team'     : [team],
        'opponent' : [opponent],
        'elo_diff' : [elo_diff],
        'form_diff': [form_diff],
    })
    return float(model.predict(df)[0])


# Sanity-Check: zwei Teams mit identischem Elo und Form
print("Sanity-Check: Germany vs Germany (identischer Elo + Form):")
_g = predict_goals('Germany', 'Germany')
_m = np.outer(poisson.pmf(range(9), _g), poisson.pmf(range(9), _g))
for _x in range(2):
    for _y in range(2):
        _m[_x, _y] *= dixon_coles_tau(_x, _y, _g, _g, RHO)
_m /= _m.sum()
print(f"  Erwartete Tore: {_g:.2f}")
print(f"  Sieg:          {float(np.sum(np.tril(_m,-1))):.1%}")
print(f"  Unentschieden: {float(np.sum(np.diag(_m))):.1%}  (Erwartung: 20-30%)")
print(f"  Niederlage:    {float(np.sum(np.triu(_m, 1))):.1%}")

Sanity-Check: Germany vs Germany (identischer Elo + Form):
  Erwartete Tore: 1.65
  Sieg:          39.2%
  Unentschieden: 21.6%  (Erwartung: 20-30%)
  Niederlage:    39.2%


In [60]:
def dc_matrix(team1, team2, max_goals=8):
    """Ergebnis-Wahrscheinlichkeitsmatrix mit Dixon-Coles-Korrektur (RHO=0.07)."""
    g1 = predict_goals(team1, team2)
    g2 = predict_goals(team2, team1)

    matrix = np.outer(
        poisson.pmf(range(max_goals + 1), g1),
        poisson.pmf(range(max_goals + 1), g2)
    )

    # Dixon-Coles-Korrektur fuer Niedrigtor-Felder
    for x in range(2):
        for y in range(2):
            matrix[x, y] *= dixon_coles_tau(x, y, g1, g2, RHO)

    # Renormieren
    matrix /= matrix.sum()
    return matrix, g1, g2


def match_probs(team1, team2):
    """Sieg / Unentschieden / Niederlage mit Dixon-Coles-Korrektur."""
    matrix, g1, g2 = dc_matrix(team1, team2)
    win1 = float(np.sum(np.tril(matrix, -1)))
    draw = float(np.sum(np.diag(matrix)))
    win2 = float(np.sum(np.triu(matrix,  1)))
    label_draw = 'Unentschieden'
    print(f"{team1:<20} Sieg:  {win1:.1%}")
    print(f"{label_draw:<20}        {draw:.1%}")
    print(f"{team2:<20} Sieg:  {win2:.1%}")
    print(f"\nErwartete Tore: {team1} {g1:.2f} – {g2:.2f} {team2}")
    print(f"Elo:            {get_elo(team1):.0f} – {get_elo(team2):.0f}")


def top_results(team1, team2, max_goals=8):
    """Die 10 wahrscheinlichsten Ergebnisse mit Dixon-Coles-Korrektur."""
    matrix, _, _ = dc_matrix(team1, team2, max_goals)
    rows = [
        (i, j, matrix[i, j])
        for i in range(max_goals + 1)
        for j in range(max_goals + 1)
    ]
    rows.sort(key=lambda x: x[2], reverse=True)
    result = pd.DataFrame(rows[:10], columns=[team1, team2, 'Wahrscheinlichkeit'])
    result[[team1, team2]] = result[[team1, team2]].astype(int)
    result['Wahrscheinlichkeit'] = result['Wahrscheinlichkeit'].map('{:.2%}'.format)
    return result

## 9 · Beispiele

In [61]:
top_results('France', 'Germany')

,France,Germany,Wahrscheinlichkeit
0,1,1,11.24%
1,1,0,10.12%
2,0,1,9.24%
3,2,1,8.69%
4,1,2,7.87%
5,2,0,6.67%
6,2,2,5.66%
7,0,0,5.60%
8,0,2,5.47%
9,3,1,4.17%


In [62]:
match_probs('France', 'France')

France               Sieg:  36.7%
Unentschieden               26.7%
France               Sieg:  36.7%

Erwartete Tore: France 1.14 – 1.14 France
Elo:            1898 – 1898


In [63]:
# Vergleich mehrerer Paarungen
for t1, t2 in [('Brazil', 'Argentina'), ('Spain', 'England'), ('Germany', 'Portugal')]:
    print(f"\n{'='*45}")
    match_probs(t1, t2)


Brazil               Sieg:  32.5%
Unentschieden               28.9%
Argentina            Sieg:  38.6%

Erwartete Tore: Brazil 0.94 – 1.06 Argentina
Elo:            1854 – 1929

Spain                Sieg:  38.4%
Unentschieden               27.5%
England              Sieg:  34.0%

Erwartete Tore: Spain 1.12 – 1.04 England
Elo:            1943 – 1868

Germany              Sieg:  39.7%
Unentschieden               22.3%
Portugal             Sieg:  38.0%

Erwartete Tore: Germany 1.58 – 1.54 Portugal
Elo:            1833 – 1860


## 10 · Nächste Ausbaustufe (Version 4)

- Automatische WM-Teilnehmerliste 2026 (48 Teams, 3 Gruppen à 4 Teams)
- Gruppenphase simulieren (Monte-Carlo, n=100.000)
- K.O.-Phase simulieren mit Elfmeter-Wahrscheinlichkeit
- Wettquoten-Features zur Kalibrierung
- Backtesting gegen historische WM-Ergebnisse